# Black Pearl Restaurant Analysis: Python Replication

This notebook reproduces the core statistical results in the original 2024 report using the final 233-restaurant dataset. It also audits sample versions, effect-size definitions, model assumptions, and claims that the original report could not support.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analysis import run_analysis

# Rebuild every result and figure from the source workbook.
artifacts = run_analysis(ROOT, build_notebook=False)
data = pd.read_csv(ROOT / 'data/processed/black_pearl_clean.csv')
print(f'Loaded {len(data)} restaurants and {len(data.columns)} cleaned fields.')

Loaded 233 restaurants and 21 cleaned fields.


## 1. Data audit

The final source contains 233 restaurants. The workbook's old regression-output sheets used only the first 153 observations, while the coefficients printed in the paper came from all 233 observations. The `hello` field is exactly favourites divided by 1,000 and is retained only for provenance. Three city-name keys require manual review.

In [2]:
quality = pd.read_csv(ROOT / 'results/data_quality_summary.csv')
print(quality.to_string(index=False))

                      check  value
            restaurant_rows    233
                    regions      6
                city_labels     23
 cuisine_labels_non_missing     29
            missing_cuisine      1
         three_diamond_rows      5
     duplicate_name_records     10
duplicate_city_name_records      6
            non_empty_notes     21


## 2. Descriptive results

The published means for environment score (4.697), favourites (68,135), reviews (3,043), and years open (7.08) all match the 233-row dataset.

![Score distributions](../figures/score_distributions.png)

## 3. Sample-version reconciliation

The table below explains why the historical workbook output and the final paper differ.

In [3]:
comparison = pd.read_csv(ROOT / 'results/regression_sample_comparison.csv')
print(comparison.round(6).to_string(index=False))

              term  coefficient_first_153  coefficient_all_233
         Intercept               0.236223             0.181651
       taste_score               0.595556             0.617355
 environment_score               0.249235             0.196327
     service_score               0.100453             0.143576
         R-squared               0.867259             0.892459
Adjusted R-squared               0.864586             0.891050


## 4. Core OLS replication

The response is the public platform total score. Predictors are the public taste, environment, and service subscores. These estimates reproduce the formula in the paper. They do **not** reveal the confidential Black Pearl jury selection weights.

In [4]:
regression = pd.read_csv(ROOT / 'results/core_regression.csv')
print(regression.round(6).to_string(index=False))

             term  coefficient  standard_error    t_stat  p_value  ci_95_low  ci_95_high  hc3_standard_error  hc3_t_stat  hc3_p_value
        Intercept     0.181651        0.109168  1.663959 0.097488  -0.033451    0.396752            0.151157    1.201732     0.230708
      taste_score     0.617355        0.031556 19.563991 0.000000   0.555179    0.679532            0.046837   13.180815     0.000000
environment_score     0.196327        0.040173  4.887084 0.000002   0.117171    0.275482            0.040488    4.849052     0.000002
    service_score     0.143576        0.043204  3.323222 0.001036   0.058448    0.228704            0.073254    1.959987     0.051210


The fitted equation is:

$$\widehat{TotalScore}=0.1817+0.6174(Taste)+0.1963(Environment)+0.1436(Service)$$

Model $R^2=0.8925$ and adjusted $R^2=0.8911$. The paper called the three slopes standardized coefficients, but they are unstandardized slopes.

![Regression coefficients](../figures/regression_coefficients.png)

In [5]:
betas = pd.read_csv(ROOT / 'results/standardized_betas.csv')
vif = pd.read_csv(ROOT / 'results/vif.csv')
print('Standardized coefficients')
print(betas.round(6).to_string(index=False))
print('\nVariance inflation factors')
print(vif.round(4).to_string(index=False))

Standardized coefficients
             term  standardized_beta
      taste_score           0.676093
environment_score           0.188645
    service_score           0.150796

Variance inflation factors
             term  auxiliary_r_squared    vif
      taste_score               0.6068 2.5431
environment_score               0.6848 3.1729
    service_score               0.7719 4.3845


The VIF values are below 5, although service has the highest shared variance. Because the outcome and predictors are rounded platform ratings, diagnostics should be interpreted cautiously. HC3 robust standard errors are included in the result table.

![Residual diagnostics](../figures/residual_diagnostics.png)

## 5. Award-count analysis

In [6]:
awards = pd.read_csv(ROOT / 'results/award_tests.csv')
print(awards.round(6).to_string(index=False))

                         metric       value
               annual_star_mean    3.493562
       black_pearl_listing_mean    3.948498
         mean_paired_difference   -0.454936
       median_paired_difference    0.000000
             wilcoxon_statistic 4438.000000
               wilcoxon_p_value    0.000015
      rank_biserial_correlation   -0.374753
                   spearman_rho    0.709216
               spearman_p_value    0.000000
               paired_cohens_dz   -0.294993
paper_style_pooled_d_not_paired   -0.246410


The paired Wilcoxon test is significant and the counts are strongly associated (Spearman $\rho=0.709$). The paper's $d=0.246$ can be reproduced only by using an independent-samples pooled standard deviation; paired $d_z$ is about 0.295 in magnitude.

![Award relationship](../figures/award_relationship.png)

## 6. Cuisine and region

In [7]:
cuisine_region = pd.read_csv(ROOT / 'results/cuisine_region_test.csv')
print(cuisine_region.round(6).to_string(index=False))

                    metric       value
     complete_observations  232.000000
               chi_squared  439.872416
           degrees_freedom  140.000000
        asymptotic_p_value    0.000000
                 cramers_v    0.615792
            expected_cells  174.000000
    expected_cells_below_5  169.000000
expected_cells_below_5_pct   97.126437
              permutations 5000.000000
       permutation_p_value    0.000200


Cramér's V reproduces the paper (about 0.616), but 97.1% of expected cells are below 5. The asymptotic chi-square p-value should not be trusted on its own, so this project adds a fixed-seed permutation test. Cuisine names are also geographically encoded, which makes part of the association structural rather than surprising.

## 7. Regional price differences and diamond-level limitation

In [8]:
regional = pd.read_csv(ROOT / 'results/region_price_omnibus.csv')
diamond = pd.read_csv(ROOT / 'results/diamond_summary.csv')
print('Regional price test')
print(regional.round(6).to_string(index=False))
print('\nDiamond-level summary')
print(diamond.round(4).to_string(index=False))

Regional price test
           metric     value
        kruskal_h 39.416443
  kruskal_p_value  0.000000
number_of_regions  6.000000

Diamond-level summary
 diamond_level  observations  mean_total_score  sd_total_score  anova_f  anova_p_value
             1           198            4.5955          0.1965   0.1672         0.8461
             2            30            4.5867          0.1592   0.1672         0.8461
             3             5            4.6400          0.1342   0.1672         0.8461


Only five observations are three-diamond restaurants, so a three-class prediction model would be unstable. A simple one-way ANOVA gives $p=0.846$, not the 0.489 value stated in the report; the earlier mixed-model specification was not preserved.

![Regional prices](../figures/price_by_region.png)

## Conclusion

The Python analysis successfully reproduces the paper's main 233-row regression and descriptive statistics. The strongest defensible conclusion is that taste has the largest conditional association with the public total rating among listed restaurants. The analysis does not recover Black Pearl's confidential selection algorithm, and the diamond-level sample is too imbalanced for a reliable multiclass prediction model.